In [1]:
SYSTEM_PROMPT = """
You are a data generation assistant for an Indonesian UMKM CRM classifier.
Generate realistic customer DM messages in Bahasa Indonesia (informal/colloquial).
A Transitional customer: has low-to-medium commitment, is still comparing options,
asking follow-up questions, or warming up after a prior interaction.
They are NOT clearly loyal (Communal) and NOT just complaining (Transactional).

Output ONLY a JSON array. No preamble, no markdown fences.
"""

ARCHETYPES = {
    "price_comparison": """
Generate {n} Transitional customer DMs where the customer is asking about price
and explicitly or implicitly comparing with another seller.
Example tone: "Kak harga X berapa ya? Soalnya tadi liat tempat lain agak beda..."
Output: [{{"content": "...", "label": 1}}]
""",
    "repeat_enquirer": """
Generate {n} Transitional customer DMs where the customer has bought before (1-2x)
and is asking about a new or related product. Shows some familiarity but not committed.
Example tone: "Kemarin beli di sini enak kak, kalau yang varian B ada ga?"
Output: [{{"content": "...", "label": 1}}]
""",
    "conditional_buyer": """
Generate {n} Transitional customer DMs where the customer wants to buy
but adds a condition (promo, ready stock, minimum order, delivery area).
Example tone: "Kalau beli 3 bisa dikurangin ongkirnya ga kak?"
Output: [{{"content": "...", "label": 1}}]
""",
    "slow_negotiation": """
Generate {n} Transitional multi-turn DM snippets (2-3 messages from customer side only)
showing a customer slowly negotiating price or terms, not yet committing.
Example tone: "Oke, terus kalau bayar dp dulu bisa? ... Ntar saya konfirm ya kak"
Output: [{{"content": "...", "label": 1}}]
""",
    "dormant_reactivation": """
Generate {n} Transitional customer DMs from a customer who hasn't bought in a while
and is just checking back in or asking if a product is still available.
Example tone: "Masih jual yang dulu itu ga kak? Udah lama ga order"
Output: [{{"content": "...", "label": 1}}]
"""
}

In [2]:
from google import genai
from dotenv import load_dotenv, find_dotenv

In [3]:
load_dotenv(find_dotenv())

True

In [4]:
model="gemini-2.5-flash-lite"

In [5]:
client = genai.Client()
response = client.models.generate_content(
    model=model, 
    contents="Say hello"
)
print(response.text)

Hello! How can I help you today?


In [ ]:
BATCH_SIZE = 50
ARCHETYPE_TARGETS = {
    "price_comparison":    720,
    "repeat_enquirer":     1200,
    "conditional_buyer":   1000,
    "slow_negotiation":     800,
    "dormant_reactivation": 500,
}

In [7]:
import json
import time

In [8]:
# TEST
prompt_template = ARCHETYPES["price_comparison"]
user_prompt = prompt_template.format(n=1)
response = client.models.generate_content(
  model=model,
  config=genai.types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    response_mime_type="application/json",
    temperature=0.9
  ),
  contents=user_prompt
)

raw = response.text
batch = json.loads(raw)


In [9]:
print(batch)

[{'content': 'Halo kak, mau nanya dong. Untuk produk A ini harganya berapa ya? Soalnya kemarin aku liat di toko sebelah kok harganya beda dikit. Boleh info detailnya?', 'label': 1}]


In [10]:
from google.genai import errors
import sys

In [11]:
all_samples = []
MAX_RETRIES = 5

# Create/clear a checkpoint file
checkpoint_file = "transitional_checkpoint.jsonl"
with open(checkpoint_file, "w") as f:
    pass

for archetype, total in ARCHETYPE_TARGETS.items():
    collected = []
    prompt_template = ARCHETYPES[archetype]

    while len(collected) < total:
        n = min(BATCH_SIZE, total - len(collected))
        user_prompt = prompt_template.format(n=n)
        retries = 0

        while retries < MAX_RETRIES:
            try:
                response = client.models.generate_content(
                    model=model,
                    config=genai.types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        response_mime_type="application/json",
                        temperature=0.9
                    ),
                    contents=user_prompt
                )
                break  # success
            except errors.APIError as e:
                if e.code == 429: # Resource Exhausted / Rate Limited
                    retries += 1
                    wait = 15 * retries  # incremental backoff
                    print(f"# 429 - rate limited, retry {retries}/{MAX_RETRIES} in {wait}s")
                    time.sleep(wait)
                elif e.code == 503: # Service Unavailable
                    retries += 1
                    print(f"# 503 - model overloaded, retry {retries}/{MAX_RETRIES}")
                    time.sleep(30)
                else:
                    # If it's a 400 (Bad Request) or other error, fail out
                    print(f"# Unhandled API Error: {e.code} - {e.message}")
                    raise e
        else:
            print(f"# Max retries hit for {archetype}, aborting.")
            sys.exit(1)

        try:
            if not response.text:
                print("# Empty response (likely safety filter) — skipping batch")
                continue
            batch = json.loads(response.text)
        except json.JSONDecodeError as e:
            print(f"# JSON parse error: {e} — skipping batch")
            time.sleep(15)
            continue

        if isinstance(batch, list):
            samples = batch
        else:
            #Convert to list
            samples = next((v for v in batch.values() if isinstance(v, list)), [])

        valid = [
            s for s in samples
            if isinstance(s.get("content"), str)
            and len(s["content"]) > 15
            and s.get("label") == 1
        ]

        collected.extend(valid)
        all_samples.extend(valid)
        with open(checkpoint_file, "a", encoding="utf-8") as f:
            for item in valid:
                f.write(json.dumps(item) + "\n")
        print(f"{archetype}: {len(collected)}/{total}")
        time.sleep(15)


price_comparison: 49/1500
price_comparison: 98/1500
price_comparison: 147/1500
price_comparison: 196/1500
price_comparison: 244/1500
# 503 - model overloaded, retry 1/5
price_comparison: 292/1500
price_comparison: 342/1500
price_comparison: 390/1500
price_comparison: 439/1500
price_comparison: 488/1500
price_comparison: 537/1500
price_comparison: 586/1500
price_comparison: 634/1500
price_comparison: 682/1500
price_comparison: 730/1500
price_comparison: 780/1500
# JSON parse error: Extra data: line 53 column 1 (char 7380) — skipping batch
# 429 - rate limited, retry 1/5 in 15s
# 429 - rate limited, retry 2/5 in 30s
# 429 - rate limited, retry 3/5 in 45s
# 429 - rate limited, retry 4/5 in 60s
# 429 - rate limited, retry 5/5 in 75s
# Max retries hit for price_comparison, aborting.


SystemExit: 1

c:\Users\lenovo\Documents\GitHub\AI_TAHAP_2\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [17]:
print(all_samples)

[]


In [16]:
# save
import pandas as pd
df_new = pd.DataFrame(all_samples)
df_new.to_csv("transitional_synthetic_5000.csv", index=False)
print(f"Total generated: {len(df_new)}")

Total generated: 0
